### Data preparation

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv('../data/course_lead_scoring_2026.csv')
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [3]:
df.dtypes

lead_source                  object
industry                     object
employment_status            object
location                     object
annual_income               float64
number_of_courses_viewed      int64
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

In [4]:
# Missing values
df.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [5]:
# Handle missing values
# Categorical columns: fill with 'NA'
# Numerical columns: fill with 0.0

categorical_cols = df.select_dtypes(include=['object']).columns
numerical_cols = df.select_dtypes(include=['float64']).columns

print("Categorical columns:", list(categorical_cols))
print("Numerical columns:", list(numerical_cols))

df[categorical_cols] = df[categorical_cols].fillna('NA')
df[numerical_cols] = df[numerical_cols].fillna(0.0)

# Verify no missing values remain
print("\nMissing values after imputation:")
print(df.isnull().sum())

Categorical columns: ['lead_source', 'industry', 'employment_status', 'location']
Numerical columns: ['annual_income', 'lead_score']

Missing values after imputation:
lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64


### Q1: What is the most frequent observation (mode) for the column industry?

In [6]:
df['industry'].mode()

0    technology
Name: industry, dtype: object

In [7]:
df['industry'].value_counts()

industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64

**Answer**: Technology is the most frequent observation with 1173 values.

### Q2: What are the two features that have the biggest correlation?

In [8]:
# Correlation matrix for numerical features
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns
corr_matrix = df[numerical_cols].corr()
corr_matrix

,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
annual_income,1.000000,0.161300,0.122842,0.229496,0.189652
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204,0.395038
interaction_count,0.122842,0.721609,1.000000,0.915746,0.448624
lead_score,0.229496,0.757204,0.915746,1.000000,0.483246
converted,0.189652,0.395038,0.448624,0.483246,1.000000


In [9]:
# Correlation for specific pairs
pairs = [
    ('interaction_count', 'lead_score'),
    ('number_of_courses_viewed', 'lead_score'),
    ('number_of_courses_viewed', 'interaction_count'),
    ('annual_income', 'interaction_count')
]

for col1, col2 in pairs:
    corr = df[col1].corr(df[col2])
    print(f"{col1} - {col2}: {corr:.4f}")

interaction_count - lead_score: 0.9157
number_of_courses_viewed - lead_score: 0.7572
number_of_courses_viewed - interaction_count: 0.7216
annual_income - interaction_count: 0.1228


**Answer**: Biggest correlation is between interaction count and lead score.

In [10]:
# Split the data
from sklearn.model_selection import train_test_split

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

In [11]:
# Remove target 'converted' from dataframes
y_train = df_train['converted']
y_val = df_val['converted']
y_test = df_test['converted']

df_train = df_train.drop('converted', axis=1)
df_val = df_val.drop('converted', axis=1)
df_test = df_test.drop('converted', axis=1)

print(f"Train: {df_train.shape}")
print(f"Val: {df_val.shape}")
print(f"Test: {df_test.shape}")

Train: (3000, 8)
Val: (1000, 8)
Test: (1000, 8)


### Q3: Which of variables `industry`, `location`, `lead_source` and `employment_status` has the biggest mutual information score?

In [12]:
from sklearn.metrics import mutual_info_score

In [13]:
# Compute mutual information between each categorical feature and target 'converted'
# using sklearn.metrics.mutual_info_score 

def mutual_info_converted_score(series):
    return mutual_info_score(series, y_train)

cat_cols = ['industry', 'location', 'lead_source', 'employment_status']

mi = df_train[cat_cols].apply(mutual_info_converted_score)
mi.sort_values(ascending=False)

lead_source          0.034346
employment_status    0.018966
industry             0.001995
location             0.001043
dtype: float64

**Answer**: lead_source has the highest mutual information value.

### Q4: Accuracy of the validation dataset?

In [14]:
# Train Logistic Regression with One-Hot Encoding
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# One-hot encode categorical variables

df_train_encoded = pd.get_dummies(df_train, columns=cat_cols, drop_first=True)
df_val_encoded = pd.get_dummies(df_val, columns=cat_cols, drop_first=True)

# Align columns (in case some categories only appear in train or val)
df_train_encoded, df_val_encoded = df_train_encoded.align(df_val_encoded, join='left', axis=1, fill_value=0)

# Train model
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(df_train_encoded, y_train)

# Predict and calculate accuracy
y_pred = model.predict(df_val_encoded)
accuracy = accuracy_score(y_val, y_pred)
round(accuracy, 2)

0.65

**Answer**: Accuracy of validation dataset is 0.65.

### Q5: Which of following feature has the smallest difference? `lead_source`, `number_of_courses_viewed` or `interaction_count?`
Note: The difference doesn't have to be positive.


In [15]:
# Feature elimination for these 3 features only
test_features = ['lead_source', 'number_of_courses_viewed', 'interaction_count']

drops = {}
for feat in test_features:
    cols_to_drop = [c for c in df_train_encoded.columns if c.startswith(feat)]
    acc = accuracy_score(
        y_val,
        LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
        .fit(df_train_encoded.drop(columns=cols_to_drop), y_train)
        .predict(df_val_encoded.drop(columns=cols_to_drop))
    )
    drops[feat] = accuracy - acc # accuracy as defined in Q4

drops
min(drops, key=drops.get)

'number_of_courses_viewed'

**Answer**: number_of_courses_viewed makes least difference.

### Q6: Which of these C leads to the best accuracy on the validation set?
    0.000001
    0.00001
    0.0001
    0.001

Note: If there are multiple options, select the smallest C.


In [16]:
# Regularized Logistic Regression: find best C
C_values = [0.000001, 0.00001, 0.0001, 0.001]

results = {}
for C in C_values:
    model = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    model.fit(df_train_encoded, y_train)
    acc = accuracy_score(y_val, model.predict(df_val_encoded))
    results[C] = round(acc, 3)

results
max(results, key=results.get)

0.001

**Answer**: Best (and smallest) C value is 0.001.